# 03 — Agent Traces

Send queries to a standalone agent loop and see the full decision trace — every tool call, arguments, results, and final response.

Uses a lightweight version of the production agent loop with in-memory tool adapters.

In [ ]:
import sys, os, json
from pathlib import Path

REPO_ROOT = Path.cwd().parents[1] if "workbench" in str(Path.cwd()) else Path.cwd()
sys.path.insert(0, str(REPO_ROOT / "src" / "backend"))

from dotenv import load_dotenv
load_dotenv(REPO_ROOT / "workbench" / ".env")

import anthropic
from IPython.display import HTML, display

# Load test data for in-memory tools
samples_path = REPO_ROOT / "workbench" / "data" / "sample-videos.json"
SAMPLE_VIDEOS = json.loads(samples_path.read_text())
print(f"Loaded {len(SAMPLE_VIDEOS)} sample videos for in-memory tools")

In [ ]:
# Import tool definitions and system prompt from production code
from app.services.agent import get_tool_definitions
from app.services.prompts import build_system_prompt
from app.services.ontology import format_ontology_for_prompt

TOOLS = get_tool_definitions()
print(f"Loaded {len(TOOLS)} tool definitions:")
for t in TOOLS:
    print(f"  - {t['name']}")

In [ ]:
def mock_tool_dispatch(tool_name: str, tool_input: dict) -> str:
    """In-memory tool adapters that work without a database."""
    if tool_name == "query_items":
        # Simple in-memory filter against sample videos
        results = SAMPLE_VIDEOS[:]
        if tool_input.get("topic"):
            results = [v for v in results if v.get("expected", {}).get("topic") == tool_input["topic"]]
        if tool_input.get("affect"):
            results = [v for v in results if v.get("expected", {}).get("affect") == tool_input["affect"]]
        if tool_input.get("search_text"):
            q = tool_input["search_text"].lower()
            results = [v for v in results if q in (v.get("caption", "") or "").lower()]
        limit = tool_input.get("limit", 20)
        return json.dumps({"success": True, "data": results[:limit], "total": len(results)})
    
    elif tool_name == "get_stats":
        return json.dumps({"success": True, "data": {
            "total_items": len(SAMPLE_VIDEOS),
            "note": "In-memory mock — stats from sample-videos.json"
        }})
    
    else:
        return json.dumps({"success": False, "error": f"Tool '{tool_name}' not available in workbench mode"})


def format_trace_html(trace: list[dict]) -> str:
    """Render a trace as color-coded HTML."""
    html = ['<div style="font-family: monospace; font-size: 13px; line-height: 1.6;">']
    colors = {
        "system": "#9C9890", "user": "#2C2926", "assistant": "#1C1B18",
        "tool_use": "#A06840", "tool_result": "#3D7A4A",
    }
    for step in trace:
        kind = step["type"]
        color = colors.get(kind, "#666")
        content = step["content"]
        if isinstance(content, dict):
            content = json.dumps(content, indent=2)
        content_escaped = str(content).replace("<", "&lt;").replace(">", "&gt;").replace("\n", "<br>")
        html.append(f'<div style="margin: 8px 0; padding: 8px; border-left: 3px solid {color}; background: #F8F7F4;">')
        html.append(f'<b style="color: {color};">[{kind.upper()}]</b><br>{content_escaped}</div>')
    html.append('</div>')
    return '\n'.join(html)

In [ ]:
async def run_traced_agent(query: str, system_prompt: str | None = None, max_turns: int = 10) -> list[dict]:
    """Run the agent loop with full trace capture."""
    client = anthropic.AsyncAnthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
    
    if system_prompt is None:
        system_prompt = (
            "You are Attic, a personal content intelligence assistant. "
            "You help users explore their TikTok history.\n\n"
            f"Ontology:\n{format_ontology_for_prompt()}"
        )
    
    trace = [{"type": "system", "content": system_prompt[:200] + "..."}]
    trace.append({"type": "user", "content": query})
    
    messages = [{"role": "user", "content": query}]
    
    for turn in range(max_turns):
        response = await client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=4096,
            system=system_prompt,
            tools=TOOLS,
            messages=messages,
        )
        
        # Collect text and tool_use blocks
        tool_uses = []
        for block in response.content:
            if block.type == "text":
                trace.append({"type": "assistant", "content": block.text})
            elif block.type == "tool_use":
                trace.append({"type": "tool_use", "content": {"name": block.name, "input": block.input}})
                tool_uses.append(block)
        
        if not tool_uses:
            break  # No more tool calls — done
        
        # Dispatch tools and feed results back
        messages.append({"role": "assistant", "content": response.content})
        tool_results = []
        for tu in tool_uses:
            result_str = mock_tool_dispatch(tu.name, tu.input)
            trace.append({"type": "tool_result", "content": {"tool": tu.name, "result": json.loads(result_str)}})
            tool_results.append({"type": "tool_result", "tool_use_id": tu.id, "content": result_str})
        messages.append({"role": "user", "content": tool_results})
    
    return trace

In [ ]:
# Run a single query and display the full trace
trace = await run_traced_agent("What kinds of cooking videos have I saved?")
display(HTML(format_trace_html(trace)))

In [ ]:
# Compare: run the same query with a modified system prompt
custom_prompt = (
    "You are Attic, a personal content intelligence assistant. "
    "Be extremely concise. Use bullet points. Never exceed 3 sentences.\n\n"
    f"Ontology:\n{format_ontology_for_prompt()}"
)

trace_v2 = await run_traced_agent("What kinds of cooking videos have I saved?", system_prompt=custom_prompt)

print("=== ORIGINAL PROMPT ===")
display(HTML(format_trace_html(trace)))
print("\n=== MODIFIED PROMPT ===")
display(HTML(format_trace_html(trace_v2)))